# AADS-ULoRA v5.5 - Google Colab Bootstrap Notebook

This notebook sets up the complete Colab environment for training the AADS-ULoRA system.

## What this notebook does:
1. **Detects GPU** - Identifies GPU type (A100, T4, etc.) and CUDA version
2. **Mounts Google Drive** - With retry logic and error handling
3. **Creates directory structure** - Sets up workspace on Drive
4. **Configures environment** - Creates Colab-specific configuration
5. **Sets up symlinks** - Optimizes path handling for Colab
6. **Validates setup** - Checks all components are working

---
**Expected Runtime:** 2-5 minutes (excluding Drive mounting)
**Required:** Google Drive with at least 50GB free space

---

## ⚠️ Important Notes
- **Do not skip any cells** - Each cell depends on previous setup
- **Wait for completion** - Let each cell finish before running the next
- **Check outputs** - Green checkmarks indicate success
- **If you encounter errors:** Check the troubleshooting section at the end

---

## 1. GPU Detection and System Information

First, we'll detect what GPU is available in your Colab runtime and gather system information.

**Why this matters:**
- Different GPUs (A100, T4, P100) have different memory capacities
- We'll adjust configuration based on your GPU type
- CUDA version determines compatible PyTorch builds

**Expected outputs:**
- GPU type (e.g., Tesla A100, Tesla T4)
- GPU memory in GB
- CUDA version
- Number of GPUs available

In [ ]:
import subprocess
import sys
import json
from pathlib import Path

def detect_gpu():
    """Detect GPU type, memory, and CUDA version."""
    print("🔍 Detecting GPU...")
    
    # Check if CUDA is available
    try:
        import torch
        cuda_available = torch.cuda.is_available()
    except ImportError:
        cuda_available = False
        print("⚠️  PyTorch not installed yet. Will check after installation.")
        return {
            'available': False,
            'type': 'Unknown',
            'memory_gb': 0,
            'cuda_version': 'Unknown',
            'device_count': 0
        }
    
    if not cuda_available:
        print("❌ CUDA is not available! This notebook requires a GPU.")
        print("   Please go to Runtime → Change runtime type → Select 'GPU'")
        return {
            'available': False,
            'type': 'None',
            'memory_gb': 0,
            'cuda_version': torch.version.cuda if hasattr(torch, 'version') else 'Unknown',
            'device_count': 0
        }
    
    # Get GPU information
    device_count = torch.cuda.device_count()
    device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(device)
    
    # Get GPU memory using nvidia-smi
    try:
        smi_output = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            encoding='utf-8'
        )
        memory_mb = int(smi_output.strip())
        memory_gb = memory_mb / 1024
    except Exception as e:
        print(f"⚠️  Could not query GPU memory: {e}")
        memory_gb = 0
    
    cuda_version = torch.version.cuda if hasattr(torch, 'version') else 'Unknown'
    
    result = {
        'available': True,
        'type': device_name,
        'memory_gb': round(memory_gb, 1),
        'cuda_version': cuda_version,
        'device_count': device_count
    }
    
    print(f"✅ GPU Detected: {result['type']}")
    print(f"   Memory: {result['memory_gb']} GB")
    print(f"   CUDA: {result['cuda_version']}")
    print(f"   Devices: {result['device_count']}")
    
    return result

# Run detection
gpu_info = detect_gpu()

# Display summary
print("\n" + "="*50)
print("SYSTEM SUMMARY")
print("="*50)
print(f"GPU Available: {gpu_info['available']}")
if gpu_info['available']:
    print(f"GPU Type: {gpu_info['type']}")
    print(f"GPU Memory: {gpu_info['memory_gb']} GB")
    print(f"CUDA Version: {gpu_info['cuda_version']}")
    
    # Provide recommendations based on GPU
    if 'A100' in gpu_info['type']:
        print("\n💡 Recommendation: A100 detected! You can use larger batch sizes (32-64).")
    elif 'T4' in gpu_info['type']:
        print("\n💡 Recommendation: T4 detected. Use smaller batch sizes (8-16).")
    elif 'P100' in gpu_info['type']:
        print("\n💡 Recommendation: P100 detected. Use moderate batch sizes (16-32).")
    else:
        print("\n💡 Recommendation: Unknown GPU. Start with batch size 16 and adjust.")
else:
    print("\n❌ WARNING: No GPU detected! Training will be extremely slow.")
    print("   Please enable GPU in Runtime → Change runtime type")
print("="*50)

## 2. Google Drive Mounting with Retry Logic

Mount your Google Drive to persist data and models.

**Why we mount Drive:**
- Colab runtimes are temporary (12 hour limit)
- All data, models, and checkpoints must be saved to Drive
- Allows resuming training after disconnection

**Mount point:** `/content/drive/MyDrive/AADS-ULoRA`

**Retry logic:**
- Up to 3 attempts with exponential backoff
- Handles network timeouts gracefully
- Validates mount success before proceeding

In [ ]:
import time
import os
from pathlib import Path
import subprocess

def mount_google_drive_with_retry(max_attempts=3, base_delay=2):
    """
    Mount Google Drive with retry logic.
    
    Args:
        max_attempts: Maximum number of retry attempts
        base_delay: Base delay in seconds (exponential backoff)
    
    Returns:
        bool: True if mount successful, False otherwise
    """
    print("🔄 Mounting Google Drive...")
    
    # Check if already mounted
    if Path('/content/drive/MyDrive').exists():
        print("✅ Google Drive already mounted")
        return True
    
    # Try to mount with retries
    for attempt in range(1, max_attempts + 1):
        try:
            print(f"   Attempt {attempt}/{max_attempts}...")
            
            # Import google.colab drive module
            from google.colab import drive
            
            # Mount with force=True to handle previous mounts
            drive.mount('/content/drive', force_remount=True)
            
            # Verify mount
            if Path('/content/drive/MyDrive').exists():
                print("✅ Google Drive mounted successfully")
                return True
            else:
                print("⚠️  Mount completed but Drive path not found")
                
        except Exception as e:
            print(f"❌ Mount attempt {attempt} failed: {e}")
            
            if attempt < max_attempts:
                delay = base_delay * (2 ** (attempt - 1))
                print(f"   Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print("❌ All mount attempts failed!")
                print("\nTroubleshooting:")
                print("1. Make sure you're signed into Google Drive")
                print("2. Check your internet connection")
                print("3. Try Runtime → Restart runtime and run again")
                return False
    
    return False

# Execute mount
mount_success = mount_google_drive_with_retry()

if not mount_success:
    raise RuntimeError("Failed to mount Google Drive. Cannot proceed.")

## 3. Create Directory Structure

Create the necessary directory structure on Google Drive.

**Directory structure:**
- `/content/drive/MyDrive/AADS-ULoRA/`
  - `data/` - Dataset storage
  - `models/` - Pretrained models
  - `outputs/` - Training outputs and logs
  - `checkpoints/` - Model checkpoints
  - `config/` - Configuration files
  - `cache/` - Temporary cache (local)
  - `temp/` - Temporary files (local)

**Note:** The `cache/` and `temp/` directories are created locally on Colab's SSD for faster I/O.

In [ ]:
def create_directory_structure():
    """Create the complete directory structure for AADS-ULoRA."""
    print("📁 Creating directory structure...")
    
    # Define paths
    drive_base = Path('/content/drive/MyDrive/AADS-ULoRA')
    local_base = Path('/content/AADS-ULoRA')
    
    # Drive directories (persistent)
    drive_dirs = [
        drive_base / 'data',
        drive_base / 'models',
        drive_base / 'outputs',
        drive_base / 'checkpoints',
        drive_base / 'config'
    ]
    
    # Local directories (temporary, faster)
    local_dirs = [
        local_base,
        local_base / 'cache',
        local_base / 'temp'
    ]
    
    # Create all directories
    all_dirs = drive_dirs + local_dirs
    
    for directory in all_dirs:
        try:
            directory.mkdir(parents=True, exist_ok=True)
            print(f"   ✅ Created: {directory}")
        except Exception as e:
            print(f"   ❌ Failed to create {directory}: {e}")
            raise
    
    print("\n✅ Directory structure created successfully")
    return {
        'drive_base': str(drive_base),
        'local_base': str(local_base),
        'drive_dirs': [str(d) for d in drive_dirs],
        'local_dirs': [str(d) for d in local_dirs]
    }

# Create directories
dirs_info = create_directory_structure()

# Check Drive space
import shutil
try:
    total, used, free = shutil.disk_usage('/content/drive/MyDrive')
    free_gb = free / (1024**3)
    print(f"\n💾 Drive Space Info:")
    print(f"   Total: {total/(1024**3):.1f} GB")
    print(f"   Used: {used/(1024**3):.1f} GB")
    print(f"   Free: {free_gb:.1f} GB")
    
    if free_gb < 50:
        print("\n⚠️  WARNING: Less than 50GB free on Drive!")
        print("   Training may fail due to insufficient space.")
        print("   Consider freeing up space or upgrading Drive storage.")
    else:
        print("\n✅ Sufficient Drive space available")
except Exception as e:
    print(f"⚠️  Could not check Drive space: {e}")

## 4. Colab-Specific Configuration

Create a Colab-specific configuration file that adapts to your environment.

**Configuration includes:**
- GPU-specific optimizations
- Path mappings for Colab environment
- Training parameters optimized for Colab
- Resource limits based on available GPU memory

The configuration will be saved to: `config/colab.json`

In [ ]:
import json
from datetime import datetime

def generate_colab_config(gpu_info, dirs_info):
    """Generate Colab-specific configuration based on GPU and environment."""
    print("⚙️  Generating Colab configuration...")
    
    # Determine batch size based on GPU memory
    memory_gb = gpu_info['memory_gb']
    
    if memory_gb >= 40:  # A100 40GB
        batch_size = 32
        gradient_accumulation = 1
        num_workers = 2
        print("   🎯 Configuring for A100 (40GB)")
    elif memory_gb >= 16:  # T4 or P100
        batch_size = 16
        gradient_accumulation = 2
        num_workers = 2
        print("   🎯 Configuring for T4/P100 (16GB)")
    else:
        batch_size = 8
        gradient_accumulation = 4
        num_workers = 1
        print("   🎯 Configuring for low-memory GPU (<16GB)")
    
    # Create configuration
    config = {
        "version": "5.5.3-colab",
        "created_at": datetime.now().isoformat(),
        "colab": {
            "mount_point": "/content/drive/MyDrive/AADS-ULoRA",
            "workspace_dir": "/content/AADS-ULoRA",
            "cache_dir": "/content/AADS-ULoRA/cache",
            "temp_dir": "/content/AADS-ULoRA/temp",
            "gpu_info": gpu_info
        },
        "paths": {
            "data_root": str(Path(dirs_info['drive_base']) / 'data'),
            "models_root": str(Path(dirs_info['drive_base']) / 'models'),
            "outputs_root": str(Path(dirs_info['drive_base']) / 'outputs'),
            "checkpoints_root": str(Path(dirs_info['drive_base']) / 'checkpoints'),
            "config_root": str(Path(dirs_info['drive_base']) / 'config')
        },
        "training": {
            "use_mixed_precision": True,
            "gradient_accumulation_steps": gradient_accumulation,
            "pin_memory": True,
            "num_workers": num_workers,
            "prefetch_factor": 2,
            "persistent_workers": True,
            "batch_size": batch_size,
            "default_epochs": 100
        },
        "optimization": {
            "cudnn_benchmark": True,
            "allow_tf32": True,
            "gradient_checkpointing": memory_gb < 24,  # Enable for GPUs with <24GB
            "empty_cache_periodically": True
        },
        "checkpointing": {
            "save_every_epoch": True,
            "keep_last_n": 5,
            "save_to_drive": True,
            "backup_local": True
        },
        "recovery": {
            "enable_oom_recovery": True,
            "max_oom_retries": 3,
            "enable_drive_monitoring": True,
            "session_timeout_seconds": 300
        }
    }
    
    return config

# Generate configuration
colab_config = generate_colab_config(gpu_info, dirs_info)

# Save to file
config_path = Path(dirs_info['drive_dirs'][3]) / 'colab.json'  # config directory
with open(config_path, 'w') as f:
    json.dump(colab_config, f, indent=2)

print(f"\n✅ Configuration saved to: {config_path}")
print(f"\n📋 Configuration Summary:")
print(f"   Batch size: {colab_config['training']['batch_size']}")
print(f"   Gradient accumulation: {colab_config['training']['gradient_accumulation_steps']}")
print(f"   Workers: {colab_config['training']['num_workers']}")
print(f"   Mixed precision: {colab_config['training']['use_mixed_precision']}")
print(f"   Gradient checkpointing: {colab_config['optimization']['gradient_checkpointing']}")

## 5. Path Handling and Symlinks

Create symbolic links to simplify path management in Colab.

**Why symlinks?**
- Shortens long paths (e.g., `/content/drive/MyDrive/AADS-ULoRA/data` → `/content/data`)
- Makes code more portable between Colab and local environments
- Reduces path length issues in some tools

**Symlinks created:**
- `/content/data` → `/content/drive/MyDrive/AADS-ULoRA/data`
- `/content/models` → `/content/drive/MyDrive/AADS-ULoRA/models`
- `/content/outputs` → `/content/drive/MyDrive/AADS-ULoRA/outputs`
- `/content/checkpoints` → `/content/drive/MyDrive/AADS-ULoRA/checkpoints`

**Note:** If symlinks already exist, they'll be recreated to ensure correctness.

In [ ]:
def create_symlinks(dirs_info):
    """Create symbolic links for easier path access."""
    print("🔗 Creating symbolic links...")
    
    drive_base = Path(dirs_info['drive_base'])
    
    # Define symlinks: local_path -> drive_path
    symlinks = {
        Path('/content/data'): drive_base / 'data',
        Path('/content/models'): drive_base / 'models',
        Path('/content/outputs'): drive_base / 'outputs',
        Path('/content/checkpoints'): drive_base / 'checkpoints'
    }
    
    created = 0
    skipped = 0
    
    for local_path, target_path in symlinks.items():
        try:
            # Remove existing symlink or directory
            if local_path.exists() or local_path.is_symlink():
                if local_path.is_dir() and not local_path.is_symlink():
                    # It's a real directory, not a symlink - skip
                    print(f"   ⚠️  {local_path} exists as directory, skipping")
                    skipped += 1
                    continue
                else:
                    # Remove symlink or file
                    local_path.unlink()
            
            # Create parent directory if needed
            local_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Create symlink
            os.symlink(target_path, local_path, target_is_directory=True)
            print(f"   ✅ Created: {local_path} → {target_path}")
            created += 1
            
        except Exception as e:
            print(f"   ❌ Failed to create symlink {local_path}: {e}")
            raise
    
    print(f"\n✅ Created {created} symlinks, skipped {skipped} existing")
    return created

# Create symlinks
create_symlinks(dirs_info)

# Verify symlinks
print("\n🔍 Verifying symlinks...")
symlink_paths = ['/content/data', '/content/models', '/content/outputs', '/content/checkpoints']
for path in symlink_paths:
    p = Path(path)
    if p.exists() and p.is_symlink():
        target = os.readlink(path)
        print(f"   ✅ {path} → {target}")
    else:
        print(f"   ❌ {path} is not a valid symlink")

## 6. Environment Variables and Path Setup

Set up environment variables and Python path for the project.

**What we're doing:**
- Adding project root to Python path
- Setting environment variables for configuration
- Configuring PyTorch/CUDA optimizations

These settings will be active for the current Colab session.

In [ ]:
import os
import sys
from pathlib import Path

def setup_environment_variables(dirs_info, colab_config):
    """Set up environment variables for Colab."""
    print("🌍 Setting up environment variables...")
    
    # Add project root to Python path
    project_root = '/content/AADS-ULoRA'
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        print(f"   ✅ Added to PYTHONPATH: {project_root}")
    
    # Set environment variables
    env_vars = {
        'AADS_ULORA_ROOT': project_root,
        'AADS_ULORA_CONFIG': str(Path(colab_config['paths']['config_root']) / 'colab.json'),
        'AADS_ULORA_DATA_ROOT': colab_config['paths']['data_root'],
        'AADS_ULORA_MODELS_ROOT': colab_config['paths']['models_root'],
        'AADS_ULORA_OUTPUTS_ROOT': colab_config['paths']['outputs_root'],
        'AADS_ULORA_CHECKPOINTS_ROOT': colab_config['paths']['checkpoints_root'],
        'CUDA_LAUNCH_BLOCKING': '0',  # Async CUDA launches for better performance
        'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:128'  # Reduce memory fragmentation
    }
    
    for key, value in env_vars.items():
        os.environ[key] = value
        print(f"   ✅ Set {key}={value}")
    
    print("\n✅ Environment variables configured")
    return env_vars

# Setup environment
env_vars = setup_environment_variables(dirs_info, colab_config)

# Configure PyTorch optimizations
print("\n⚡ Configuring PyTorch optimizations...")
try:
    import torch
    
    # Set cuDNN benchmark (enables auto-tuner for better performance)
    if colab_config['optimization']['cudnn_benchmark']:
        torch.backends.cudnn.benchmark = True
        print("   ✅ cuDNN benchmark enabled")
    
    # Enable TF32 on Ampere+ GPUs (A100)
    if colab_config['optimization']['allow_tf32'] and hasattr(torch.backends.cuda, 'matmul'):
        torch.backends.cuda.matmul.allow_tf32 = True
        print("   ✅ TF32 enabled")
    
    print("\n✅ PyTorch optimizations configured")
except ImportError:
    print("   ⚠️  PyTorch not yet installed - optimizations will be applied after installation")

## 7. Validation and Testing

Validate that all components are set up correctly.

**Checks performed:**
- Directory structure exists
- Symlinks are working
- Configuration file is valid JSON
- Environment variables are set
- GPU is accessible (if PyTorch installed)

If all checks pass, you're ready to proceed to the next notebook!

In [ ]:
def validate_setup(dirs_info, colab_config):
    """Validate the complete setup."""
    print("✅ VALIDATING SETUP")
    print("="*50)
    
    errors = []
    warnings = []
    
    # 1. Check Drive directories
    print("\n1. Checking Drive directories...")
    for directory in dirs_info['drive_dirs']:
        if Path(directory).exists():
            print(f"   ✅ {directory}")
        else:
            print(f"   ❌ Missing: {directory}")
            errors.append(f"Missing directory: {directory}")
    
    # 2. Check local directories
    print("\n2. Checking local directories...")
    for directory in dirs_info['local_dirs']:
        if Path(directory).exists():
            print(f"   ✅ {directory}")
        else:
            print(f"   ❌ Missing: {directory}")
            errors.append(f"Missing local directory: {directory}")
    
    # 3. Check symlinks
    print("\n3. Checking symlinks...")
    symlink_paths = ['/content/data', '/content/models', '/content/outputs', '/content/checkpoints']
    for path in symlink_paths:
        p = Path(path)
        if p.exists() and p.is_symlink():
            target = os.readlink(path)
            print(f"   ✅ {path} → {target}")
        else:
            print(f"   ❌ {path} is not a valid symlink")
            errors.append(f"Invalid symlink: {path}")
    
    # 4. Check configuration file
    print("\n4. Checking configuration file...")
    config_file = Path(dirs_info['drive_dirs'][3]) / 'colab.json'  # config directory
    if config_file.exists():
        try:
            with open(config_file, 'r') as f:
                config = json.load(f)
            print(f"   ✅ Configuration file is valid JSON")
            print(f"   📋 Version: {config.get('version', 'Unknown')}")
            print(f"   📋 GPU type: {config['colab']['gpu_info']['type']}")
        except json.JSONDecodeError as e:
            print(f"   ❌ Configuration file is invalid JSON: {e}")
            errors.append("Invalid configuration JSON")
    else:
        print(f"   ❌ Configuration file not found: {config_file}")
        errors.append("Missing configuration file")
    
    # 5. Check environment variables
    print("\n5. Checking environment variables...")
    required_env_vars = [
        'AADS_ULORA_ROOT',
        'AADS_ULORA_CONFIG',
        'AADS_ULORA_DATA_ROOT'
    ]
    for var in required_env_vars:
        if var in os.environ:
            print(f"   ✅ {var}={os.environ[var]}")
        else:
            print(f"   ❌ Missing: {var}")
            errors.append(f"Missing environment variable: {var}")
    
    # 6. Check PyTorch and CUDA (if available)
    print("\n6. Checking PyTorch and CUDA...")
    try:
        import torch
        print(f"   ✅ PyTorch {torch.__version__} installed")
        
        if torch.cuda.is_available():
            print(f"   ✅ CUDA available: {torch.version.cuda}")
            print(f"   ✅ GPU devices: {torch.cuda.device_count()}")
            print(f"   ✅ Current device: {torch.cuda.current_device()}")
        else:
            print(f"   ⚠️  CUDA not available (GPU not accessible)")
            warnings.append("CUDA not available - check GPU runtime")
    except ImportError:
        print("   ⚠️  PyTorch not installed yet")
        warnings.append("PyTorch not installed - run installation next")
    
    # Summary
    print("\n" + "="*50)
    print("VALIDATION SUMMARY")
    print("="*50)
    
    if errors:
        print(f"\n❌ {len(errors)} ERROR(S) FOUND:")
        for i, error in enumerate(errors, 1):
            print(f"   {i}. {error}")
    else:
        print("\n✅ No errors found")
    
    if warnings:
        print(f"\n⚠️  {len(warnings)} WARNING(S):")
        for i, warning in enumerate(warnings, 1):
            print(f"   {i}. {warning}")
    
    if not errors:
        print("\n🎉 Setup validation PASSED!")
        print("\nNext steps:")
        print("1. Run the installation notebook (if not already done)")
        print("2. Proceed to data preparation notebook")
        print("3. Start training!")
        return True
    else:
        print("\n❌ Setup validation FAILED!")
        print("\nPlease fix the errors above before proceeding.")
        return False

# Run validation
validation_passed = validate_setup(dirs_info, colab_config)

# Final summary
print("\n" + "="*50)
print("BOOTSTRAP COMPLETE")
print("="*50)
if validation_passed:
    print("✅ Colab environment is ready!")
    print("\nYou can now proceed to the next notebook in the sequence:")
    print("  1_data_preparation.ipynb")
else:
    print("❌ Setup incomplete. Please resolve errors above.")

## 8. Troubleshooting Guide

Common issues and solutions:

### Issue: Google Drive won't mount
**Solutions:**
1. Make sure you're signed into Google Drive in another tab
2. Try Runtime → Restart runtime, then run this notebook again
3. Check your internet connection
4. If using multiple Google accounts, try signing out of all and sign in with just one

### Issue: No GPU detected
**Solutions:**
1. Go to Runtime → Change runtime type
2. Select 'GPU' as hardware accelerator
3. Click 'Save' and restart runtime
4. Run this notebook again

### Issue: Out of memory on Drive
**Solutions:**
1. Check free space: `!df -h` in a code cell
2. Delete old checkpoints from `/content/drive/MyDrive/AADS-ULoRA/checkpoints`
3. Clear Colab cache: `!rm -rf /content/AADS-ULoRA/cache/*`
4. Consider using Colab Pro for more Drive space

### Issue: Symlink creation fails
**Solutions:**
1. Colab may not allow symlinks in some environments
2. The code will still work without symlinks (using full paths)
3. Check if you have write permissions

### Issue: PyTorch installation fails
**Solutions:**
1. Check CUDA version: `!nvcc --version`
2. Install matching PyTorch version manually:
   - CUDA 11.8: `!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118`
   - CUDA 12.1: `!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121`
3. Restart runtime after installation

---

## Need More Help?

Check the documentation:
- `docs/colab_migration_guide.md`
- `docs/user_guide/colab_training_manual.md`
- `docs/cheatsheet_colab.md`

Or open an issue on GitHub with:
- Colab runtime type (Free/Pro)
- GPU type
- Error messages (full traceback)
- Steps to reproduce

In [ ]:
# Display final information
print("="*60)
print("🎯 COLAB BOOTSTRAP SUMMARY")
print("="*60)
print(f"\n📂 Project Root: {env_vars['AADS_ULORA_ROOT']}")
print(f"📂 Config File: {env_vars['AADS_ULORA_CONFIG']}")
print(f"📂 Data Directory: {env_vars['AADS_ULORA_DATA_ROOT']}")
print(f"📂 Models Directory: {env_vars['AADS_ULORA_MODELS_ROOT']}")
print(f"📂 Outputs Directory: {env_vars['AADS_ULORA_OUTPUTS_ROOT']}")
print(f"📂 Checkpoints Directory: {env_vars['AADS_ULORA_CHECKPOINTS_ROOT']}")

print("\n⚙️  Training Configuration:")
print(f"   Batch size: {colab_config['training']['batch_size']}")
print(f"   Gradient accumulation: {colab_config['training']['gradient_accumulation_steps']}")
print(f"   Workers: {colab_config['training']['num_workers']}")
print(f"   Mixed precision: {colab_config['training']['use_mixed_precision']}")
print(f"   Gradient checkpointing: {colab_config['optimization']['gradient_checkpointing']}")

print("\n💾 Storage Info:")
try:
    free_gb = shutil.disk_usage('/content/drive/MyDrive').free / (1024**3)
    print(f"   Drive free space: {free_gb:.1f} GB")
except:
    pass

print("\n✅ Bootstrap complete! Ready for installation and training.")
print("\n📝 Next steps:")
print("   1. Install dependencies (if not already done)")
print("   2. Run data preparation notebook")
print("   3. Start Phase 1 training")
print("="*60)